In [ ]:
class SmartMeter:
    """A smart energy grid that represents home utility meters,
    tracks electricity usage, rates, and billing state"""

    DEFAULT_RATE_PER_KWH = 0.15
    _total_meters_active = 0

    def __init__(self, account_id: str, customer_name: str):
        if not account_id or not customer_name:
            raise ValueError("Input cannot be empty")
        else:
            self._account_id = account_id
            self._customer_name = customer_name
            self._current_reading_kwh = 0.0
            self._unbilled_amount = 0.0
            SmartMeter._total_meters_active += 1

    @property
    def account_id(self) -> str:
        return self._account_id

    @property
    def customer_name(self) -> str:
        return self._customer_name

    @customer_name.setter
    def customer_name(self, value):
        if value and value.strip():
            self._customer_name = value
        else:
            raise ValueError("Input must be a non-empty string")

    @property
    def current_reading_kwh(self) -> float:
        return self._current_reading_kwh

    @property
    def unbilled_amount(self) -> float:
        return self._unbilled_amount

    def consume_energy(self, kwh_amount: float):
        if kwh_amount <= 0:
            raise ValueError("Input must be a positive value")
        self._current_reading_kwh += kwh_amount
        self._unbilled_amount += kwh_amount * self.DEFAULT_RATE_PER_KWH

    def generate_bill(self) -> float:
        billing = self._unbilled_amount
        self._reset_bill()
        return round(billing, 2)

    def _reset_bill(self):
        self._unbilled_amount = 0.0

    @classmethod
    def get_active_meters_count(cls) -> int:
        return cls._total_meters_active

In [ ]:
class InventoryItem:
    """E-Commerce Inventory Manager"""

    TAX_RATE = 0.08
    _total_catalog_items = 0

    def __init__(self,
                 sku: str,
                 name: str,
                 base_price: float,
                 stock_quantity: int,
                 reorder_threshold: int = 5
                 ):
        if not sku or not name or base_price <= 0 or stock_quantity < 0 or reorder_threshold < 0:
            raise ValueError("Invalid Input")
        self._sku = sku
        self._name = name
        self._base_price = base_price
        self._stock_quantity = stock_quantity
        self._reorder_threshold = reorder_threshold
        InventoryItem._total_catalog_items += 1

    @property
    def sku(self):
        return self._sku

    @property
    def base_price(self):
        return self._base_price

    @base_price.setter
    def base_price(self, price: float):
        if price <= 0:
            raise ValueError("Price must be a positive value")
        self._base_price = price

    @property
    def stock_quantity(self):
        return self._stock_quantity

    @property
    def reorder_threshold(self):
        return self._reorder_threshold

    @reorder_threshold.setter
    def reorder_threshold(self, threshold: int):
        if threshold < 0:
            raise ValueError("Threshold must be a positive integer")
        self._reorder_threshold = threshold

    def restock(self, quantity: int):
        if quantity <= 0:
            raise ValueError("Input must be a positive integer")
        self._stock_quantity += quantity

    def sell(self, quantity: int) -> float:
        if quantity > self._stock_quantity or quantity <= 0:
            raise ValueError("Insufficient stock or Invalid Purchase")
        self._stock_quantity -= quantity
        return (quantity * self._base_price) * (1 + self.TAX_RATE)

    def is_reorder_needed(self):
        if self._stock_quantity <= self._reorder_threshold:
            return True
        return False


In [ ]:
class SecureBankAccount:
    """A banking backend system"""

    MAX_DAILY_TRANSFER_LIMIT = 5000
    OVERDRAFT_FEE = 35.0

    def __init__(self, account_number: str, initial_balance: float = 0.0):
        if initial_balance < 0:
            raise ValueError("Input cannot be negative")
        self._account_number = account_number
        self._balance = initial_balance
        self._transferred_today = 0.0
        self._is_frozen = False

    @property
    def account_number(self):
        return self._account_number

    @property
    def balance(self):
        return self._balance

    @property
    def is_frozen(self):
        return self._is_frozen

    def freez_account(self):
        self._is_frozen = True

    def unfreez_account(self):
        self._is_frozen = False

    def deposit(self, amount):
        if self._is_frozen or amount <= 0:
            raise ValueError("Account frozen or invalid deposit amount")
        self._balance += amount

    def withdraw(self, amount):
        if self._is_frozen or amount <=0:
            raise ValueError("Account frozen or invalid withdrawal amount")

        if amount > self._balance:
            balance_after_withdrawal = self._balance - amount - self.OVERDRAFT_FEE
            if balance_after_withdrawal >= -500.0:
                self._balance = balance_after_withdrawal
            else:
                raise ValueError("Balance cannot be less than $500.00")
        else:
            self._balance -= amount

    def transfer(self, target_account: 'SecureBankAccount', amount: float):
        if target_account._is_frozen or self._is_frozen:
            raise ValueError("At least one account is frozen")
        
        if (self._transferred_today + amount) <= self.MAX_DAILY_TRANSFER_LIMIT:
            self.withdraw(amount)
            target_account.deposit(amount)
            self._transferred_today += amount
        else:
            raise ValueError("Maximun daily transfer limit hit")

    def reset_daily_transfer(self):
        self._transferred_today = 0.0


In [ ]:
class SensorNode:
    """IoT Sensor Array Node"""

    CRITICAL_TEMP_THRESHOLD = 85.0
    MAX_HISTORY_BUFFER = 5

    def __init__(self, device_id: str, location: str):
        if not device_id or not device_id.strip() or not location or not location.strip():
            raise ValueError("Invalid Input")
        self._device_id = device_id
        self._location = location
        self._readings = []
        self._status = "OK"

    @property
    def device_id(self) -> str:
        return self._device_id

    @property
    def location(self) -> str:
        return self._location

    @location.setter
    def location(self, location):
        if not location or not location.strip():
            raise ValueError("Invalid input")
        self._location = location

    @property
    def status(self) -> str:
        return self._status

    @property
    def readings(self) -> list[float]:
        return self._readings.copy()

    def record_reading(self, temp_celcuis: float):
        if len(self._readings) >= self.MAX_HISTORY_BUFFER:
            self._readings.pop(0)
            self._readings.append(temp_celcuis)
        else:
            self._readings.append(temp_celcuis)

        if temp_celcuis >= self.CRITICAL_TEMP_THRESHOLD:
            self._status = "CRITICAL"
        elif 70.0 <= temp_celcuis < self.CRITICAL_TEMP_THRESHOLD:
            self._status = "WARNING"
        else:
            self._status = "OK"

    def get_average_temperature(self):
        if len(self._readings) == 0:
            return 0.0
        else:
            return sum(self._readings)/len(self._readings)


In [ ]:
class FleetVehicle:
    """Vehicle Fleet Manager"""

    MAINTENANCE_INTERNAL_KM = 10_000.0
    FUEL_CONSUMPTION_RATE = 0.08

    def __init__(
            self,
            vin: str,
            model: str,
            fuel_tank_capacity: float,
            inititial_odometer: float = 0.0
            ):
        if fuel_tank_capacity < 0 or inititial_odometer < 0:
            raise ValueError("Input must be positive")
        
        self._vin = vin
        self._model = model
        self._current_fuel_litres = fuel_tank_capacity
        self._last_service_odometer = inititial_odometer
        self._odometer_km = inititial_odometer
        self._fuel_tank_capacity = fuel_tank_capacity

    @property
    def vin(self):
        return self._vin

    @property
    def odometer_km(self):
        return self._odometer_km

    @property
    def fuel_level_percent(self):
        if self._fuel_tank_capacity > 0:
            return (self._current_fuel_litres/self._fuel_tank_capacity) * 100
        else:
            raise ValueError("Fuel percentage cannot be obtained now since current capacity is at zero")

    def drive(self, distance_km):
        if distance_km <= 0:
            raise ValueError("Distance cannot be zero or nagative")
        required_fuel = distance_km * self.FUEL_CONSUMPTION_RATE

        if required_fuel > self._current_fuel_litres:
            raise ValueError("Insufficient fuel to complete trip")
        self._current_fuel_litres -= required_fuel
        self._odometer_km += distance_km

    def refuel(self, litres):
        if litres <= 0:
            raise ValueError("Input cannot be zero or negative")

        if (litres + self._current_fuel_litres) > self._fuel_tank_capacity:
            self._current_fuel_litres = self._fuel_tank_capacity
        else:
            self._current_fuel_litres += litres

    def needs_service(self):
        if (self._odometer_km - self._last_service_odometer) >= self.MAINTENANCE_INTERNAL_KM:
            return True
        else:
            return False

    def perform_service(self):
        self._last_service_odometer = self._odometer_km
